# 📄 arXiv 論文爬蟲 — BeautifulSoup 版本
### 領域：電腦視覺 / 圖形辨識（Computer Vision & Image Recognition）

---

## 📌 這份 Notebook 的學習路徑

```
階段一：認識 arXiv API 網址長什麼樣
    ↓
階段二：用 Python 發送請求，看看伺服器回傳什麼
    ↓
階段三：認識 BeautifulSoup，了解它怎麼解析 XML
    ↓
階段四：用 BeautifulSoup 逐步取出每個欄位
    ↓
階段五：整理成函式，方便重複使用
    ↓
階段六：執行爬蟲，搜尋 CV 論文，儲存結果
```

---

## 📌 跟原版（ET）的差異

| | ET 版本 | BeautifulSoup 版本 |
|---|---|---|
| 輸入 | `response.content`（bytes） | `response.text`（字串） |
| 解析工具 | `xml.etree.ElementTree` | `BeautifulSoup` |
| 命名空間 | 需要手動宣告 `namespace` 字典 | **不需要！自動處理** |
| 取標籤語法 | `entry.find("atom:title", namespace)` | `entry.find("title")` |

---
# 階段一：認識 arXiv API 網址

在寫任何程式之前，先用**瀏覽器**直接打開下面這個網址，看看 API 回傳什麼：

👉 http://export.arxiv.org/api/query?search_query=cat:cs.CV&max_results=2

你會看到一大串 XML 文字，那就是 arXiv 給我們的論文資料。

---

### 網址的組成結構

```
http://export.arxiv.org/api/query
        ↑ 這是 API 的基本網址（Base URL）

?search_query=cat:cs.CV&max_results=2
 ↑ 問號後面是參數，用 & 分隔多個參數
```

| 參數 | 意思 | 範例值 |
|------|------|--------|
| `search_query` | 搜尋什麼 | `cat:cs.CV`（電腦視覺分類） |
| `max_results` | 要幾筆資料 | `10` |
| `start` | 從第幾筆開始 | `0`（第一頁） |
| `sortBy` | 排序方式 | `submittedDate` |
| `sortOrder` | 升冪或降冪 | `descending`（最新優先） |

---
# 階段二：用 Python 發送請求

In [ ]:
# 安裝需要的套件
# beautifulsoup4：解析 XML / HTML 的工具
# lxml：BeautifulSoup 解析 XML 時需要的底層引擎
%pip install requests pandas beautifulsoup4 lxml

In [1]:
import requests
from bs4 import BeautifulSoup   # 從 beautifulsoup4 套件載入 BeautifulSoup
import pandas as pd
import time

print("✅ 所有套件載入成功")

✅ 所有套件載入成功


In [2]:
# ▶ 發送一次請求，只要 2 筆資料
# 目的：確認網路連線正常，API 有回應

url = "http://export.arxiv.org/api/query?search_query=cat:cs.CV&max_results=2"

response = requests.get(url)

print("HTTP 狀態碼：", response.status_code)
# 200 = 成功，404 = 找不到，500 = 伺服器錯誤

HTTP 狀態碼： 200


In [3]:
# ▶ 確認 response.text 是什麼型別

print("型別：", type(response.text))
# <class 'str'>  → 就是一般的字串

print("\n前 300 個字元：")
print(response.text[:300])

型別： <class 'str'>

前 300 個字元：
<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/WFquG1CEVlfg5AL6NF3pd5hfYDo</id>
  <title>arXiv Query: search_query=cat:cs.CV&amp;id_list=


In [4]:
# ▶ 印出完整的原始 XML
# 仔細看看 XML 的結構：哪裡是標題？哪裡是摘要？哪裡是作者？

print(response.text)

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/WFquG1CEVlfg5AL6NF3pd5hfYDo</id>
  <title>arXiv Query: search_query=cat:cs.CV&amp;id_list=&amp;start=0&amp;max_results=2</title>
  <updated>2026-05-15T03:28:39Z</updated>
  <link href="https://arxiv.org/api/query?search_query=cat:cs.CV&amp;start=0&amp;max_results=2&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>2</opensearch:itemsPerPage>
  <opensearch:totalResults>192215</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2012.11486v1</id>
    <title>Leaf Segmentation and Counting with Deep Learning: on Model Certainty, Test-Time Augmentation, Trade-Offs</title>
    <updated>2020-12-21T17:00:05Z</updated>
    <link href="https://arxiv.org/abs/2012.11486v1" rel="alternate" type="text/html"/>
 

---
## 觀察 XML 的結構

執行上一格後，你會看到類似這樣的 XML：

```xml
<?xml version="1.0" encoding="UTF-8"?>
<feed xmlns="http://www.w3.org/2005/Atom">

  <!-- 每篇論文是一個 <entry> 區塊 -->
  <entry>
    <title>論文標題</title>
    <summary>論文摘要...</summary>
    <published>2024-01-15T00:00:00Z</published>
    <author><name>作者姓名</name></author>
    <id>https://arxiv.org/abs/2401.12345</id>
  </entry>

</feed>
```

所以我們的任務就是：**把每個 `<entry>` 裡面我們要的欄位取出來**

---
# 階段三：認識 BeautifulSoup

## BeautifulSoup 是什麼？

BeautifulSoup 是一個專門用來**解析 HTML 和 XML** 的套件。
它最大的優點是：**不需要處理命名空間（namespace）**，語法比 ET 更直覺。

## 使用時需要指定 parser（解析引擎）

```python
BeautifulSoup(資料, "解析引擎")
```

| Parser | 用途 | 需要安裝 |
|--------|------|----------|
| `"xml"` | 解析 XML（我們用這個） | 需要安裝 `lxml` |
| `"html.parser"` | 解析 HTML | 內建，不需安裝 |
| `"lxml"` | 解析 HTML，速度更快 | 需要安裝 `lxml` |

我們的資料是 XML 格式，所以用 `"xml"`

In [5]:
# ▶ 用 BeautifulSoup 解析 response.text
# 注意：這裡傳入的是 response.text（字串），不是 response.content（bytes）

soup = BeautifulSoup(response.text, "xml")
# soup 就像 ET 版本的 root，是整份 XML 的入口

print("解析成功！")
print("型別：", type(soup))

解析成功！
型別： <class 'bs4.BeautifulSoup'>


In [6]:
# ▶ 看看 soup 的根標籤是什麼

print("根標籤名稱：", soup.find("feed").name)
# 不需要寫 namespace！直接寫標籤名稱就好

根標籤名稱： feed


In [7]:
# ▶ 對比 ET 版本和 BeautifulSoup 版本的語法差異

# ET 版本：需要宣告 namespace，語法比較長
# namespace = {"atom": "http://www.w3.org/2005/Atom"}
# root.findall("atom:entry", namespace)
# entry.find("atom:title", namespace).text

# BeautifulSoup 版本：直接寫標籤名稱，不需要 namespace
entries = soup.find_all("entry")
print("找到幾個 entry：", len(entries))
# 直接就找到了！不需要任何 namespace 設定

找到幾個 entry： 2


---
# 階段四：用 BeautifulSoup 逐步取出每個欄位

In [8]:
# ▶ 先只看第一篇論文

first_entry = entries[0]
print("第一篇論文的標籤名稱：", first_entry.name)

第一篇論文的標籤名稱： entry


In [9]:
# ▶ 取出標題
# BeautifulSoup 用 .find() 找標籤，用 .text 取文字

title_raw = first_entry.find("title").text
print("原始標題（未處理）：", repr(title_raw))
# repr() 可以看到隱藏的換行符號 \n 和空白

title_clean = title_raw.strip()   # 去除前後空白和換行
print("清理後標題：", title_clean)

原始標題（未處理）： 'Leaf Segmentation and Counting with Deep Learning: on Model Certainty, Test-Time Augmentation, Trade-Offs'
清理後標題： Leaf Segmentation and Counting with Deep Learning: on Model Certainty, Test-Time Augmentation, Trade-Offs


In [10]:
# ▶ 取出摘要（Abstract）

summary = first_entry.find("summary").text.strip()
print("摘要（前 200 字）：")
print(summary[:200], "...")

摘要（前 200 字）：
Plant phenotyping tasks such as leaf segmentation and counting are fundamental to the study of phenotypic traits. Since it is well-suited for these tasks, deep supervised learning has been prevalent i ...


In [11]:
# ▶ 取出投稿日期

published_raw = first_entry.find("published").text
print("原始日期：", published_raw)
# 格式是 2024-01-15T00:00:00Z

published_clean = published_raw[:10]   # 只取前 10 字元
print("清理後日期：", published_clean)

原始日期： 2020-12-21T17:00:05Z
清理後日期： 2020-12-21


In [12]:
# ▶ 取出作者（可能有多位）
# find_all() 找出所有 author 標籤

author_tags = first_entry.find_all("author")
print("作者數量：", len(author_tags))

# 每個 author 標籤裡面有一個 name 標籤
authors = [a.find("name").text for a in author_tags]
print("作者列表：", authors)

作者數量： 2
作者列表： ['Douglas Pinto Sampaio Gomes', 'Lihong Zheng']


In [13]:
# ▶ 取出論文連結
# arXiv 的連結放在 <id> 標籤裡

link = first_entry.find("id").text
print("論文連結：", link)

論文連結： http://arxiv.org/abs/2012.11486v1


In [14]:
# ▶ 把第一篇論文的所有欄位整合成一個字典

paper = {
    "title"    : first_entry.find("title").text.strip(),
    "summary"  : first_entry.find("summary").text.strip(),
    "published": first_entry.find("published").text[:10],
    "authors"  : [a.find("name").text
                  for a in first_entry.find_all("author")],
    "link"     : first_entry.find("id").text
}

# 印出這個字典
for key, value in paper.items():
    print(f"\n【{key}】")
    print(value)


【title】
Leaf Segmentation and Counting with Deep Learning: on Model Certainty, Test-Time Augmentation, Trade-Offs

【summary】
Plant phenotyping tasks such as leaf segmentation and counting are fundamental to the study of phenotypic traits. Since it is well-suited for these tasks, deep supervised learning has been prevalent in recent works proposing better performing models at segmenting and counting leaves. Despite good efforts from research groups, one of the main challenges for proposing better methods is still the limitation of labelled data availability. The main efforts of the field seem to be augmenting existing limited data sets, and some aspects of the modelling process have been under-discussed. This paper explores such topics and present experiments that led to the development of the best-performing method in the Leaf Segmentation Challenge and in another external data set of Komatsuna plants. The model has competitive performance while been arguably simpler than other recent

In [15]:
# ▶ 把所有 entry 都處理（用 for 迴圈）

papers = []   # 空串列，用來存所有論文

for entry in entries:
    paper = {
        "title"    : entry.find("title").text.strip(),
        "summary"  : entry.find("summary").text.strip(),
        "published": entry.find("published").text[:10],
        "authors"  : [a.find("name").text
                      for a in entry.find_all("author")],
        "link"     : entry.find("id").text
    }
    papers.append(paper)

print(f"✅ 共取出 {len(papers)} 篇論文")
print("第一篇標題：", papers[0]["title"])
print("第二篇標題：", papers[1]["title"])

✅ 共取出 2 篇論文
第一篇標題： Leaf Segmentation and Counting with Deep Learning: on Model Certainty, Test-Time Augmentation, Trade-Offs
第二篇標題： PointINet: Point Cloud Frame Interpolation Network


---
# 階段五：把以上步驟整理成一個函式

上面的步驟每次要爬新的關鍵字都要重寫一遍，很麻煩。

所以我們把它**包成一個函式**，之後只要呼叫函式就好。

In [ ]:
def fetch_arxiv_papers(query, max_results=10):
    """
    從 arXiv API 爬取論文資料（BeautifulSoup 版本）

    參數：
        query      : 搜尋關鍵字，例如 'cat:cs.CV AND image recognition'
        max_results: 最多回傳幾筆，預設 10

    回傳：
        papers: 論文資料的串列（每篇論文是一個字典）
    """

    # --- 步驟 1：設定 API 網址與參數 ---
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": query,
        "start"       : 0,
        "max_results" : max_results,
        "sortBy"      : "submittedDate",
        "sortOrder"   : "descending"
    }

    # --- 步驟 2：發送 GET 請求，取得 response.text（字串）---
    response = requests.get(base_url, params=params)

    # --- 步驟 3：用 BeautifulSoup 解析 XML ---
    # 傳入 response.text（字串），指定用 "xml" parser
    soup = BeautifulSoup(response.text, "xml")

    # --- 步驟 4：找出所有 entry 標籤 ---
    entries = soup.find_all("entry")

    # --- 步驟 5：逐篇取出資料 ---
    papers = []
    for entry in entries:
        paper = {
            # .find() 找標籤，.text 取文字，.strip() 去空白
            "title"    : entry.find("title").text.strip(),
            "summary"  : entry.find("summary").text.strip(),
            # [:10] 只取日期部分 YYYY-MM-DD
            "published": entry.find("published").text[:10],
            # find_all() 找所有 author，取出每個的 name
            "authors"  : [a.find("name").text
                          for a in entry.find_all("author")],
            "link"     : entry.find("id").text
        }
        papers.append(paper)

    return papers

print("✅ 函式定義完成，可以開始使用")

---
# 階段六：執行爬蟲，搜尋電腦視覺論文

In [ ]:
# ▶ 單一關鍵字搜尋

papers = fetch_arxiv_papers(
    query="cat:cs.CV AND (image recognition OR object detection)",
    max_results=10
)

print(f"✅ 共爬取到 {len(papers)} 篇論文")

In [ ]:
# ▶ 轉成 DataFrame，顯示標題和日期

df = pd.DataFrame(papers)
print(df[["title", "published"]].to_string())

In [ ]:
# ▶ 在 Jupyter 裡直接顯示完整表格（會有漂亮的格線）
df

In [ ]:
# ▶ 查看第一篇論文的完整內容

first = papers[0]
print("📌 標題  ：", first["title"])
print("📅 日期  ：", first["published"])
print("👤 作者  ：", ", ".join(first["authors"]))
print("🔗 連結  ：", first["link"])
print("\n📝 摘要：")
print(first["summary"])

In [ ]:
# ▶ 多個關鍵字搜尋（這裡才需要 time.sleep）
# 多次請求之間加 3 秒延遲，避免對伺服器造成負擔

queries = [
    "cat:cs.CV AND image recognition",
    "cat:cs.CV AND object detection",
    "cat:cs.CV AND image segmentation"
]

all_papers = []

for q in queries:
    print(f"🔍 搜尋中：{q}")
    result = fetch_arxiv_papers(query=q, max_results=5)
    all_papers.extend(result)
    time.sleep(3)   # 等 3 秒再發下一次請求

# 去除重複論文（同一篇可能被多個關鍵字搜到）
df_all = pd.DataFrame(all_papers).drop_duplicates(subset="link")

print(f"\n✅ 總共爬取到 {len(df_all)} 篇不重複論文")

In [ ]:
# ▶ 儲存成 CSV
# utf-8-sig：讓 Excel 開啟時中文不會亂碼

df_all.to_csv("cv_papers_bs4.csv", index=False, encoding="utf-8-sig")
print(f"✅ 已儲存！共 {len(df_all)} 筆論文 → cv_papers_bs4.csv")